# Factor Model Demo

This notebook demonstrates the full Week 1 pipeline:
1. Load equity prices for a sample universe
2. Load Fama-French 5-factor returns
3. Convert prices to excess returns using the FRED risk-free rate
4. Fit a factor model on a rolling window
5. Inspect loadings, idiosyncratic risk, and the asset covariance matrix

All operations respect point-in-time discipline: the `as_of` parameter ensures
no future data leaks into the fit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tapt.data import (
    load_equity_prices,
    load_fama_french_factors,
    load_risk_free_rate,
)
from tapt.data.loaders import compute_returns
from tapt.factors import estimate_factor_model, to_excess_returns

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Load data

Universe: 10 large-cap US stocks across diverse sectors. Sample period: 2015-2024.

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'JPM', 'BAC', 'JNJ', 'PFE', 'XOM', 'CVX', 'PG']
start, end = '2014-12-01', '2024-12-31'

prices = load_equity_prices(tickers, start=start, end=end)
print(f'Prices shape: {prices.shape}')
print(f'Date range: {prices.index.min().date()} to {prices.index.max().date()}')
prices.tail()

In [ ]:
ff5 = load_fama_french_factors(model='5factor', frequency='monthly', start='2014-01-01', as_of='2024-12-31')
print(f'Factor returns shape: {ff5.shape}')
ff5.head()

In [ ]:
rf = load_risk_free_rate(start, end)
print(f'Risk-free rate: {rf.shape[0]} daily observations')
rf.tail()

## 2. Convert to monthly excess returns

Compute monthly returns from daily prices, then subtract the per-period risk-free rate.

In [ ]:
monthly_prices = prices.resample('ME').last()
monthly_returns = compute_returns(monthly_prices)
monthly_rf = rf.resample('ME').last()

excess_returns = to_excess_returns(
    monthly_returns,
    monthly_rf,
    rate_periods_per_year=12,
)
excess_returns.tail()

## 3. Fit the factor model

Fit on the most recent 60 months ending at 2024-12-31.

In [ ]:
fit = estimate_factor_model(
    excess_asset_returns=excess_returns,
    factor_returns=ff5,
    window=60,
    min_obs=36,
    as_of='2024-12-31',
)
fit

In [ ]:
print('Factor loadings (B):')
fit.loadings.round(2)

In [ ]:
summary = pd.DataFrame({
    'alpha (annualized)': fit.alpha * 12,
    'idio_vol (annualized)': np.sqrt(fit.idio_variance * 12),
    'r_squared': fit.r_squared,
})
summary.round(3)

## 4. Asset covariance matrix

Sigma = B * Cov(F) * B' + D. By construction this is symmetric and positive
semi-definite, with the systematic component (low rank) and idiosyncratic
component (diagonal) cleanly separated.

In [ ]:
Sigma = fit.asset_covariance()
annualized_vol = np.sqrt(np.diag(Sigma) * 12)
annualized_corr = Sigma.div(np.sqrt(np.diag(Sigma)), axis=0).div(np.sqrt(np.diag(Sigma)), axis=1)

print('Implied annualized volatility:')
print(pd.Series(annualized_vol, index=Sigma.index).round(3))
print()
print('Implied correlation matrix:')
annualized_corr.round(2)

## 5. Expected returns

Two views: factor-implied expected returns using historical factor premia, and
factor-implied plus alpha (the latter is more aggressive and prone to overfitting).

In [ ]:
mu_no_alpha = fit.expected_returns(include_alpha=False) * 12
mu_with_alpha = fit.expected_returns(include_alpha=True) * 12

comparison = pd.DataFrame({
    'Factor-implied (annualized)': mu_no_alpha,
    'Factor + alpha (annualized)': mu_with_alpha,
})
comparison.round(3)

## 6. Sanity checks

Verify the constructed covariance is well-formed.

In [ ]:
eigvals = np.linalg.eigvalsh(Sigma.values)
print(f'Min eigenvalue: {eigvals.min():.2e}')
print(f'Max eigenvalue: {eigvals.max():.2e}')
print(f'Condition number: {eigvals.max() / eigvals.min():.1f}')
print(f'PSD check: {(eigvals >= -1e-10).all()}')
print(f'Symmetry check: {np.allclose(Sigma.values, Sigma.values.T)}')

## 7. What's next

Week 2: feed this fit into a cvxpy-based optimizer to construct minimum tracking
error, mean-variance, and Black-Litterman portfolios. The covariance matrix and
expected returns from this notebook are exactly what those optimizers consume.